# 🖥️ Notebook 13: Serving Locally

Loading and running models with mlx-lm, llama.cpp Metal backend, benchmarking inference speed, and interactive chat.

**Prerequisites:** Notebooks 01-12

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Local Serving Options

| Option | Format | Strengths | Best For |
|--------|--------|-----------|----------|
| **mlx-lm** | MLX/safetensors | Native Apple Silicon, fast | Development, fine-tuning |
| **llama.cpp** | GGUF | Cross-platform, mature | Production, compatibility |
| **Direct MLX** | Custom | Full control | Research, custom models |

## mlx-lm: Load and Generate

mlx-lm loads quantized models from Hugging Face and runs them natively on Apple Silicon.

⚠️ **Memory note**: Loading a model requires sufficient unified memory. A 7B 4-bit model needs ~4GB.

In [ ]:
# mlx-lm usage (requires downloading a model)
# Uncomment to run — this will download a model from Hugging Face

# from mlx_lm import load, generate
# 
# # Load a small quantized model
# model, tokenizer = load('mlx-community/Qwen2.5-0.5B-Instruct-4bit')
# 
# # Generate text
# prompt = 'Explain transformers in one sentence:'
# response = generate(model, tokenizer, prompt=prompt, max_tokens=50)
# print(response)

print('mlx-lm usage:')
print('  from mlx_lm import load, generate')
print('  model, tokenizer = load("mlx-community/Qwen2.5-0.5B-Instruct-4bit")')
print('  response = generate(model, tokenizer, prompt="Hello", max_tokens=50)')
print()
print('⚡ mlx-lm handles quantization, KV-cache, and Metal optimization automatically')

## llama.cpp Metal Backend

llama.cpp with Metal backend provides efficient inference using GGUF model format.

In [ ]:
# llama-cpp-python usage (requires downloading a GGUF model)
# Uncomment to run

# from llama_cpp import Llama
# 
# llm = Llama(model_path='path/to/model.gguf', n_gpu_layers=-1)  # -1 = all layers on GPU
# output = llm('Explain transformers:', max_tokens=50)
# print(output['choices'][0]['text'])

print('llama-cpp-python usage:')
print('  from llama_cpp import Llama')
print('  llm = Llama(model_path="model.gguf", n_gpu_layers=-1)')
print('  output = llm("Hello", max_tokens=50)')
print()
print('💡 n_gpu_layers=-1 offloads all layers to Metal GPU')

## Performance Comparison

Expected inference speeds on M4 Pro (48GB) with 4-bit quantization:

| Model | mlx-lm (tok/s) | llama.cpp (tok/s) | Memory |
|-------|----------------|-------------------|--------|
| 3B 4-bit | ~80-100 | ~70-90 | ~2 GB |
| 7B 4-bit | ~40-60 | ~35-50 | ~4 GB |
| 12B 4-bit | ~25-35 | ~20-30 | ~7 GB |

⚡ mlx-lm is typically 10-20% faster than llama.cpp on Apple Silicon due to native Metal optimization.

## Interactive Chat

A simple chat loop with conversation history and streaming output.

In [ ]:
# Simple chat loop template (requires a loaded model)
def chat_loop(model=None, tokenizer=None):
    """Interactive chat with conversation history."""
    history = []
    system_prompt = 'You are a helpful AI assistant.'
    
    print('Chat (type "quit" to exit)')
    print('-' * 40)
    
    while True:
        user_input = input('You: ')
        if user_input.lower() == 'quit':
            break
        
        history.append({'role': 'user', 'content': user_input})
        
        # In practice, format history into a prompt and generate
        # response = generate(model, tokenizer, prompt=formatted_prompt)
        response = f'[Model response to: {user_input}]'  # placeholder
        
        history.append({'role': 'assistant', 'content': response})
        print(f'Assistant: {response}')

print('Chat loop template defined.')
print('To use: load a model with mlx-lm, then call chat_loop(model, tokenizer)')
print()
print('**Next:** Notebook 14 — Capstone: Fine-tune & Serve Gemma 4')

## Summary

| Option | Pros | Cons |
|--------|------|------|
| mlx-lm | Native Metal, fast, easy | Apple-only |
| llama.cpp | Cross-platform, mature | Slightly slower on Mac |
| Direct MLX | Full control | More code to write |

**Next:** Notebook 14 — Capstone: Fine-tune & Serve Gemma 4

## ⚡ Inference Speed vs Model Size

Bigger models know more, but they're slower — every parameter must be read from memory for each token generated. With 4-bit quantization, each parameter takes half a byte, so a 70B model still needs ~35 GB of memory bandwidth per token. Since Apple Silicon's memory bandwidth is fixed (~273 GB/s on M4 Pro), smaller models simply have fewer bytes to move and generate tokens faster.

💡 **Why quantized models are fast:** Inference speed on Apple Silicon is *memory-bandwidth bound*, not compute bound. Quantization shrinks each weight from 2 bytes (float16) to 0.5 bytes (4-bit), so the GPU can read 4× more parameters per second — directly translating to ~4× faster token generation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Approximate inference speeds on M4 Pro 48GB (4-bit quantization, mlx-lm)
model_labels = ['3B\n4-bit', '7B\n4-bit', '12B\n4-bit', '27B\n4-bit', '70B\n4-bit']
tokens_per_sec = [90, 45, 30, 15, 7]

fig, ax = plt.subplots(figsize=(8, 5))

# Bar chart
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
bars = ax.bar(model_labels, tokens_per_sec, color=colors, edgecolor='white', linewidth=1.2, width=0.6)

# Add value labels on bars
for bar, val in zip(bars, tokens_per_sec):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'~{val} tok/s', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Human reading speed reference line
ax.axhline(y=4, color='#3498db', linestyle='--', linewidth=2, label='Human reading speed (~4 tok/s)')

ax.set_ylabel('Tokens per Second', fontsize=12)
ax.set_xlabel('Model Size (4-bit Quantized)', fontsize=12)
ax.set_title('Inference Speed on M4 Pro 48GB — Smaller Models Are Faster', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('⚡ Key takeaway: even the 70B model generates faster than humans read.')
print('   Smaller quantized models are ideal for interactive use — 3B at ~90 tok/s feels instant.')

## 🧪 Try It Yourself

Try local model serving:

1. **Install Ollama**: Run `brew install ollama`, then `ollama run qwen2.5:0.5b`. Chat with it! How fast does it respond?

2. **Compare sizes**: If you have enough memory, try a 3B and 7B model. Can you feel the speed difference? The quality difference?

3. **Memory check**: While a model is running, open Activity Monitor and check memory usage. Does it match the expected model size?

## 📜 History & Alternatives

### The Evolution of Local LLM Inference

Running LLMs locally went from impossible to mainstream in just two years. The key enablers were quantization, efficient inference engines, and Apple Silicon's unified memory.

| Year | Tool / Framework | Team | Key Contribution |
|------|-----------------|------|-----------------|
| 2023 (Mar) | **llama.cpp** | Georgi Gerganov | CPU-first LLM inference in C/C++ — GGML format, 4-bit quantization, ran LLaMA on laptops |
| 2023 (Jun) | **GPTQ** | Frantar et al. | Post-training quantization to 4-bit with minimal quality loss — GPU-focused |
| 2023 (Jun) | **ExLlama** | turboderp | Optimized GPTQ inference for consumer GPUs — fast 4-bit on RTX cards |
| 2023 (Aug) | **vLLM** | UC Berkeley | PagedAttention for efficient batched serving — became the production standard |
| 2023 (Dec) | **MLX** | Apple ML Research | Native Apple Silicon ML framework — unified memory, lazy evaluation |
| 2024 (Jan) | **Ollama** | Ollama team | One-command local LLM serving — `ollama run llama3` — democratized local inference |
| 2024 (Feb) | **LM Studio** | LM Studio team | GUI for local LLMs — model discovery, chat interface, OpenAI-compatible API |
| 2024 (Mar) | **mlx-lm** | Apple + Community | High-level LLM serving on MLX — quantization, streaming, model conversion |
| 2024 (Jun) | **AWQ** | MIT HAN Lab | Activation-aware quantization — better quality than GPTQ at same bit-width |
| 2024 (Aug) | **SGLang** | UC Berkeley | Structured generation + RadixAttention — faster than vLLM for complex prompts |
| 2024 | **ExLlamaV2** | turboderp | EXL2 format — mixed quantization per layer, best quality/speed tradeoff |
| 2025 | **llama.cpp (GGUF v3)** | Community | Improved quantization (IQ formats), speculative decoding, vision support |
| 2025 | **MLX-VLM** | Apple + Community | Vision-language model serving on MLX — multimodal local inference |

### 💡 The Local Inference Stack

```
User Interface:    Ollama CLI / LM Studio GUI / Open WebUI
         ↓
API Layer:         OpenAI-compatible REST API
         ↓
Inference Engine:  llama.cpp / MLX / vLLM / SGLang
         ↓
Model Format:      GGUF / SafeTensors / EXL2
         ↓
Hardware:          Apple Silicon (UMA) / NVIDIA GPU (VRAM) / CPU (RAM)
```

### Local Inference Engine Comparison

| Engine | Best For | Hardware | Quantization | Batched Serving |
|--------|---------|----------|-------------|-----------------|
| **llama.cpp** | CPU + mixed GPU, portability | Any (CPU, CUDA, Metal) | GGUF (2-8 bit) | Basic |
| **MLX / mlx-lm** | Apple Silicon native | Apple Silicon only | 4/8-bit | Single user |
| **vLLM** | Production serving | NVIDIA GPU | GPTQ, AWQ, FP8 | Excellent (PagedAttention) |
| **SGLang** | Structured output, complex prompts | NVIDIA GPU | GPTQ, AWQ | Excellent |
| **Ollama** | Ease of use | Any (wraps llama.cpp) | GGUF | Basic |
| **LM Studio** | GUI experience | Any (wraps llama.cpp) | GGUF | Basic |
| **ExLlamaV2** | Maximum quality at low bits | NVIDIA GPU | EXL2 | Good |

### ⚡ Performance Benchmarks (Approximate)

For Llama 3.1 8B (4-bit quantized) on different platforms:
- **M4 Pro 48GB (MLX)**: ~45 tokens/sec
- **M4 Pro 48GB (llama.cpp)**: ~35 tokens/sec
- **RTX 4090 (vLLM)**: ~120 tokens/sec
- **RTX 4090 (ExLlamaV2)**: ~130 tokens/sec
- **CPU only (llama.cpp)**: ~8 tokens/sec

Apple Silicon excels at running larger models (30B-70B) that don't fit in GPU VRAM — a 64GB Mac can serve models that would need $2000+ in NVIDIA hardware.

### ⚠️ Common Pitfalls

- **VRAM vs RAM confusion**: GGUF models report "size" as file size, not runtime memory — actual usage is 1.2-1.5× file size due to context buffers and KV-cache
- **Quantization quality varies by model**: 4-bit works well for 13B+ models but noticeably degrades smaller (1-3B) models — always benchmark perplexity before deploying
- **Context length traps**: Advertised context lengths (128K+) require proportionally more memory for KV-cache — a 70B model at 128K context can exceed 48GB even quantized
- **Ollama hides complexity**: Convenient but uses default quantization and sampling settings that may not suit your use case — check `ollama show <model>` for actual parameters
- **Apple Silicon thermal throttling**: Sustained inference on laptops can hit thermal limits after 10-15 minutes, reducing throughput by 20-30% — monitor with `powermetrics`

### 🎯 Interview Tip

> *"The local inference ecosystem bifurcated along hardware lines: llama.cpp dominates CPU and cross-platform deployment with GGUF format, while vLLM dominates NVIDIA GPU serving with PagedAttention for efficient batching. MLX fills the Apple Silicon niche with native unified memory support. The key insight is that for single-user inference, memory bandwidth (not compute) is the bottleneck — which is why quantization (reducing bytes per parameter) directly translates to faster token generation. Apple Silicon's high memory bandwidth per dollar makes it surprisingly competitive for personal LLM serving."*